In [0]:
# ── Cell: Install GE ───────────────────────────────────────────
%pip install great-expectations==0.18.0 pandas pyarrow

In [0]:
dbutils.library.restartPython()

In [0]:
# ── Cell: Imports ─────────────────────────────────────────────
import great_expectations as gx
from great_expectations.core.batch import RuntimeBatchRequest
from great_expectations.data_context.types.base import (
    DataContextConfig,
    InMemoryStoreBackendDefaults
)
from datetime import datetime
import pandas as pd

print(f"GE version : {gx.__version__}")

In [0]:
# ── Cell: Load Silver table ────────────────────────────────────
TABLE_NAME = "ipl_catalog.silver.deliveries"
SUITE_NAME = "deliveries_suite"

df_silver_spark = spark.table(TABLE_NAME)
total_rows      = df_silver_spark.count()
print(f"Total rows in Silver deliveries: {total_rows:,}")

# ── Convert a sample to pandas for GE validation ──────────────
# We use a sample because pandas cannot hold 1M+ rows in memory
# 50K rows is enough to validate data quality patterns
SAMPLE_SIZE = 50_000

df_sample = (
    df_silver_spark
    .sample(fraction = SAMPLE_SIZE / total_rows, seed = 42)
    .toPandas()
)

print(f"Sample rows for GE validation : {len(df_sample):,}")
print(f"Columns                       : {len(df_sample.columns)}")

In [0]:
# ── Cell: Setup GE context ─────────────────────────────────────
context = gx.get_context(
    project_config = DataContextConfig(
        store_backend_defaults = InMemoryStoreBackendDefaults()
    )
)

# Add pandas datasource — no Spark dependency
datasource = context.sources.add_pandas(
    name = "pandas_datasource"
)

# Add dataframe asset
data_asset = datasource.add_dataframe_asset(
    name = "silver_deliveries_sample"
)

# Build batch request from pandas DataFrame
batch_request = data_asset.build_batch_request(
    dataframe = df_sample
)

print("GE context and datasource ready ✓")

In [0]:
# ── Cell: Create suite and validator ──────────────────────────
suite = context.add_or_update_expectation_suite(
    expectation_suite_name = SUITE_NAME
)

validator = context.get_validator(
    batch_request          = batch_request,
    expectation_suite_name = SUITE_NAME,
)

print("Validator ready ✓")

In [0]:
# ── Cell: Add all expectations ─────────────────────────────────

IPL_TEAMS = [
    "Mumbai Indians", "Chennai Super Kings",
    "Royal Challengers Bengaluru", "Kolkata Knight Riders",
    "Delhi Capitals",  "Rajasthan Royals",
    "Sunrisers Hyderabad", "Punjab Kings",
    "Lucknow Super Giants", "Gujarat Titans",
    "Kochi Tuskers Kerala", "Pune Warriors",
    "Rising Pune Supergiant", "Gujarat Lions",

    "Kings XI Punjab", "Delhi Daredevils", "Royal Challengers Bangalore", "Rising Pune Supergiants", "Deccan Chargers"
]

# ── Section 1: Table level ─────────────────────────────────────
validator.expect_table_row_count_to_be_between(
    min_value = 1_000,       # sample — not full 500K
    max_value = 500_000,
)
print("✓ Row count expectation added")

# ── Section 2: Not null on critical columns ────────────────────
critical_cols = [
    "match_id", "innings", "over_number", "ball_number",
    "batting_team", "bowling_team", "striker",
    "bowler", "runs_off_bat", "total_runs", "phase"
]
for col in critical_cols:
    validator.expect_column_values_to_not_be_null(column=col)
print(f"✓ Not null added for {len(critical_cols)} columns")

# ── Section 3: Range checks ────────────────────────────────────
validator.expect_column_values_to_be_between(
    column="innings", min_value=1, max_value=6
)
validator.expect_column_values_to_be_between(
    column="over_number", min_value=0, max_value=19
)
validator.expect_column_values_to_be_between(
    column="runs_off_bat", min_value=0, max_value=6
)
validator.expect_column_values_to_be_between(
    column="total_runs", min_value=0, max_value=26
)
print("✓ Range checks added")

# ── Section 4: Accepted values ────────────────────────────────
validator.expect_column_values_to_be_in_set(
    column    = "batting_team",
    value_set = IPL_TEAMS
)
validator.expect_column_values_to_be_in_set(
    column    = "bowling_team",
    value_set = IPL_TEAMS
)
validator.expect_column_values_to_be_in_set(
    column    = "phase",
    value_set = ["powerplay", "middle", "death", "super_over"]
)
validator.expect_column_values_to_be_in_set(
    column    = "_source",
    value_set = ["cricsheet"]
)
validator.expect_column_values_to_be_in_set(
    column    = "season",
    value_set = [str(y) for y in range(2007, 2027)]
)
print("✓ Accepted values added")

# ── Section 5: Boolean columns ────────────────────────────────
for col in ["is_wicket", "is_boundary", "is_six", "is_dot_ball"]:
    validator.expect_column_values_to_be_in_set(
        column    = col,
        value_set = [True, False]
    )
print("✓ Boolean checks added")

# ── Section 6: Consistency ────────────────────────────────────
validator.expect_column_pair_values_A_to_be_greater_than_B(
    column_A = "total_runs",
    column_B = "runs_off_bat",
    or_equal = True,
)
print("✓ Consistency checks added")

# ── Save suite ────────────────────────────────────────────────
validator.save_expectation_suite(
    discard_failed_expectations = False
)

total_expectations = len(
    validator.get_expectation_suite().expectations
)
print(f"\nSuite '{SUITE_NAME}' saved")
print(f"Total expectations : {total_expectations}")

In [0]:

# ── Cell: Run validation ───────────────────────────────────────
results     = validator.validate()
statistics  = results["statistics"]

total       = statistics["evaluated_expectations"]
passed      = statistics["successful_expectations"]
failed      = statistics["unsuccessful_expectations"]
success_pct = statistics["success_percent"]

print(f"Validation Results")
print(f"{'='*40}")
print(f"  Total      : {total}")
print(f"  Passed     : {passed}")
print(f"  Failed     : {failed}")
print(f"  Success %  : {success_pct:.1f}%")

if failed > 0:
    print(f"\nFailed expectations:")
    for result in results.results:
        if not result.success:
            col = result.expectation_config.kwargs.get(
                "column", "table-level"
            )
            exp = result.expectation_config.expectation_type
            print(f"  ✗ {col:30} {exp}")
else:
    print("\n✓ All expectations passed — Silver data is clean")